In [ ]:
import pymupdf
import re
import os
from tqdm import tqdm
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


In [ ]:

# --- CONFIGURATION ---
# Rename your messy PDF to this:
PDF_PATH = "char.pdf" 
# This will create a separate DB folder from your Ashtanga one
DB_PERSIST_DIRECTORY = "./chroma_db_charaka" 
EMBEDDING_MODEL_NAME = "intfloat/e5-large-v2"
BATCH_SIZE = 32


In [ ]:

class CharakaIngestor:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.embeddings = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL_NAME,
            model_kwargs={'device': 'cpu'} 
        )

    def clean_messy_text(self, text: str) -> str:
        """
        Aggressive cleaning for messy bilingual OCR scans.
        """
        # 1. Remove common headers (Adjust regex based on your specific PDF visual header)
        text = re.sub(r'(Charaka|Samhita|Vol|Page).*?\d+', '', text, flags=re.IGNORECASE)
        
        # 2. Fix broken hyphenation (e.g. "treat-\nment" -> "treatment")
        text = re.sub(r'-\s*\n\s*', '', text)
        
        # 3. Heal broken sentences:
        # If a line ends with a lowercase letter and next starts with lowercase, merge them.
        text = re.sub(r'([a-z])\n([a-z])', r'\1 \2', text)
        
        # 4. Normalize Sanskrit Dandas (Convert | and । to || for consistency)
        text = text.replace('॥', '||').replace('।', '|')
        
        # 5. Collapse excessive whitespace
        return re.sub(r'\n{3,}', '\n\n', text).strip()

    def load_and_process_pdf(self):
        if not os.path.exists(self.pdf_path):
            raise FileNotFoundError(f"❌ File not found: {self.pdf_path}")
            
        print(f"📖 Opening {self.pdf_path}...")
        doc = pymupdf.open(self.pdf_path)
        processed_docs = []
        
        # TQDM Progress Bar for Reading Pages
        for page_num, page in enumerate(tqdm(doc, desc="Reading Pages", unit="pg")):
            text = self.clean_messy_text(page.get_text())
            
            # Skip empty pages
            if len(text) < 50: continue

            processed_docs.append(Document(
                page_content=text,
                metadata={
                    "page": page_num + 1,
                    "source": "Charaka_Samhita_Chikitsa",
                    "type": "Therapeutics"
                }
            ))
            
        print(f"✅ Extracted {len(processed_docs)} valid pages.")
        return processed_docs

    def chunk_documents(self, documents):
        print("✂️  Chunking (Priority: Sanskrit Verses)...")
        
        # We prioritize '||' so the Sanskrit Shloka stays with the English translation
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1200, 
            chunk_overlap=200,
            separators=[
                "||",             # Double Danda (Highest priority)
                "\n\n",           # Paragraphs
                ". ",             # Sentences
                " "               # Words
            ],
            keep_separator=True   # Keep the || in the text
        )
        
        chunks = text_splitter.split_documents(documents)
        print(f"🧩 Split into {len(chunks)} chunks.")
        return chunks

    def create_vector_db(self, chunks):
        print(f"⚙️  Creating Database at '{DB_PERSIST_DIRECTORY}'...")
        vector_store = Chroma(
            embedding_function=self.embeddings,
            persist_directory=DB_PERSIST_DIRECTORY
        )
        
        # Create batches
        batches = [chunks[i:i + BATCH_SIZE] for i in range(0, len(chunks), BATCH_SIZE)]
        
        # TQDM Progress Bar for Embeddings
        for batch in tqdm(batches, desc="Embedding Batches", unit="batch"):
            vector_store.add_documents(documents=batch)

        print("✅ Charaka Database Ready!")


In [ ]:

ingestor = CharakaIngestor(PDF_PATH)
raw_docs = ingestor.load_and_process_pdf()
chunks = ingestor.chunk_documents(raw_docs)
ingestor.create_vector_db(chunks)